# Episode 5 — Pytrees

**Student workbook** · code along with the video.

JAX treats nested **dict / list / tuple** structures of arrays as **pytrees**. Model parameters, batches, and optimizer state all use this pattern — and utilities like **`jax.tree.map`** let you transform every leaf at once.

| | |
|---|---|
| **Chapter** | 1.5 · Part I — Pure JAX |
| **Prereq** | Episodes 1–4 |
| **Next** | [Episode 6 — GPT-2 Transformer](../ep06/student.ipynb) |

**Source:** [Pytrees](https://docs.jax.dev/en/latest/pytrees.html) · [`jax.tree.map`](https://docs.jax.dev/en/latest/_autosummary/jax.tree.map.html) · [`jax.tree.leaves`](https://docs.jax.dev/en/latest/_autosummary/jax.tree.leaves.html)


In [ ]:
# your code here


## What is a pytree?

A pytree is built from **nodes** (`list`, `tuple`, `dict`) and **leaves** (arrays, scalars, or anything not registered as a container). JAX walks the nodes and collects the leaves — that's how `grad`, `jit`, and `vmap` handle whole parameter trees.

In ML you'll see pytrees everywhere: model weights, dataset rows, RL observations. `jax.tree.leaves` is the quick way to count or inspect them.


In [ ]:
# your code here


Lists, tuples, and dicts are in JAX's built-in registry. Custom classes can be registered too — see [Custom pytree nodes](https://docs.jax.dev/en/latest/custom_pytrees.html) when you outgrow plain dicts.


### `NamedTuple` is a pytree

`collections.namedtuple` (and `typing.NamedTuple`) register as container nodes — fields are flattened by **attribute name**, same as in the key-path examples later.

In [ ]:
# your code here


## `jax.tree.map`

The workhorse pytree utility. Like Python's `map`, but applied leaf-by-leaf across an entire tree. Pass two or more trees and the function receives matching leaves from each — **structures must match** (same keys, same list lengths).


In [ ]:
# your code here


In [ ]:
# your code here


## MLP params + SGD update

A common pattern: store params as a **list of layer dicts**, inspect shapes with `tree.map`, then update with SGD. `jax.grad` returns a pytree with the **same structure** as `params` — one `tree.map` subtracts the learning-rate-scaled gradient from every weight and bias.


In [ ]:
# your code here


In [ ]:
# your code here


In [ ]:
# your code here


## `tree_structure`

When debugging, `tree_structure` shows how JAX classifies an object — useful when something you expect to be a leaf is actually treated as a node (or vice versa).


In [ ]:
# your code here


## Pytrees and transforms

Transformations like `vmap` and `scan` accept pytrees of arrays. Their axis arguments (`in_axes`, `out_axes`) can be pytrees too, aligned with the argument structure.

You can also pass a **prefix** — a single value broadcast to an entire subtree:

```python
vmap(f, in_axes=(None, {"k1": None, "k2": 0}))  # only map over k2
vmap(f, in_axes=(None, 0))                         # map over whole dict
vmap(f, in_axes=0)                                 # map everything (default)
```


## Key paths

Every leaf has a unique **key path** — the sequence of indices/keys from root to leaf. Handy for debugging which parameter is which:

- `tree_flatten_with_path` — flatten with paths
- `tree_map_with_path` — map with paths passed to the function
- `keystr` — pretty-print a path like `['layer']['weights']`


In [ ]:
# your code here


In [ ]:
# your code here


## Gotchas

### Shapes are tuples (nodes), not leaves

An array's `.shape` is a Python tuple — a pytree **node**. Mapping `jnp.ones` over shapes calls it on each dimension separately, not on the pair `(2, 3)`. Fix: avoid the intermediate map, or wrap the shape in `jnp.array` to make it a leaf.


In [ ]:
# your code here


### `None` is absent, not a leaf

By default, `None` entries are **skipped** — not counted as leaves. Pass `is_leaf=lambda x: x is None` when you need to preserve them.


In [ ]:
# your code here


### Dict keys must be sortable

JAX flattens dicts by **sorted keys** (for stable structure / JIT caching). Mixing `int` and `str` keys that can't be ordered raises an error. Use `OrderedDict` or a custom pytree node if you need a different key order.


In [ ]:
# your code here


## Transpose: list of trees → tree of lists

Common when batching dataset steps: turn `[{t: 1, obs: 3}, {t: 2, obs: 4}]` into `{t: [1, 2], obs: [3, 4]}`.

**Option 1:** `tree.map` with a variadic lambda (simple). **Option 2:** `tree.transpose` when you need explicit inner/outer structure control.


In [ ]:
# your code here


In [ ]:
# your code here


For custom containers (Flax `Module`, dataclasses, etc.), register flatten/unflatten handlers — see [Custom pytree nodes](https://docs.jax.dev/en/latest/custom_pytrees.html).

---

**Next:** [Episode 6 — GPT-2 Transformer](../ep06/student.ipynb).
